# 17 - Multiprocessing


## 1. Multiprocessing Fundamentals (Parallelism & Separate GIL)

`multiprocessing` modülü, her biri kendi Python yorumlayıcısına ve kendi bellek alanına (memory space) sahip bağımsız işletim sistemi süreçleri (processes) oluşturarak CPU yoğun (CPU-bound) görevlerin çok çekirdekli işlemcilerde gerçek paralel (parallel) olarak çalışmasını sağlar.

**Syntax**

```python
import multiprocessing as mp

cpu_count: int = mp.cpu_count()
```

**Temel Kavramlar ve Karşılaştırma**

* **Concurrency vs. Parallelism:** `asyncio` ve standart `threading` eşzamanlılık (concurrency) sunar; görevler tek çekirdek üzerinde sırayla geçiş yaparak ilerler. `multiprocessing` ise görevleri farklı CPU çekirdeklerine dağıtarak aynı anda gerçek paralellik sağlar.
* **Bağımsız GIL:** Her alt süreç kendi Global Interpreter Lock (GIL) kilidine sahip olduğu için standart CPython'da dahi CPU çekirdeklerinin tamamı aynı anda kullanılabilir.
* **Süreç Maliyeti (Overhead):** Yeni bir süreç başlatmak ve süreçler arası veri taşımak thread oluşturmaya kıyasla daha maliyetlidir; bu nedenle kısa G/Ç (I/O) beklemeleri yerine ağır hesaplamalarda tercih edilir.


In [ ]:
import multiprocessing as mp
from collections.abc import Callable
from datetime import datetime
from functools import wraps
from time import perf_counter
from typing import ParamSpec, TypeVar

P = ParamSpec('P')
R = TypeVar('R')


def measure_time(func: Callable[P, R]) -> Callable[P, R]:
    @wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        start: float = perf_counter()
        result: R = func(*args, **kwargs)
        elapsed: float = perf_counter() - start
        print(f'Elapsed ({func.__name__}): {elapsed:.4f}s')
        return result

    return wrapper


def current_timestamp() -> str:
    return datetime.now().strftime('%H:%M:%S')


def cpu_burn(iterations: int = 25_000_000) -> int:
    total: int = 0
    for i in range(iterations):
        total += i & 1
    return total


print(f'Available CPU cores: {mp.cpu_count()}')
print(f'Current time: {current_timestamp()}')


## 2. Creating & Managing Processes (`mp.Process`)

`multiprocessing.Process`, tıpkı `threading.Thread` arayüzüne benzer şekilde hedef bir fonksiyonu bağımsız bir işletim sistemi süreci olarak başlatmayı ve tamamlanmasını beklemeyi sağlar.

**Syntax**

```Python
p = mp.Process(target=worker_fn, name='Worker-1', args=(arg1,), kwargs={'key': val})
p.start()
p.join()
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **Giriş Noktası Koruması (`if __name__ == '__main__':`):** Özellikle Windows ve macOS (`spawn` başlangıç yöntemi) üzerinde alt süreçlerin ana modülü sonsuz döngüde yeniden içe aktarmasını önlemek için süreç başlatma kodları mutlaka ana blok içinde korunmalıdır.
* **Jupyter / IPython Notu:** Etkileşimli notebook hücrelerinde tanımlanan fonksiyonlar `spawn` tabanlı sistemlerde serileştirilemeyebilir (`PicklingError`); Linux üzerinde `fork` bağlamı (`mp.get_context('fork')`) doğrudan çalışırken taşınabilirlik için hedef fonksiyonların ayrı bir `.py` modülünde tutulması önerilir.
* **Süreç Kimliği:** Her alt süreç işletim sisteminden benzersiz bir PID (`os.getpid()`) ve `mp.current_process().name` alır.


In [ ]:
import multiprocessing as mp
import os


def run_worker(task_label: str, iterations: int) -> None:
    print(f'Started {mp.current_process().name} (PID: {os.getpid()}) -> {task_label}')
    cpu_burn(iterations)
    print(f'Finished {mp.current_process().name} (PID: {os.getpid()})')


@measure_time
def run_two_processes() -> None:
    ctx = mp.get_context('fork') if 'fork' in mp.get_all_start_methods() else mp.get_context()
    p1 = ctx.Process(target=run_worker, name='Compute-1', args=('Batch-A', 20_000_000))
    p2 = ctx.Process(target=run_worker, name='Compute-2', args=('Batch-B', 20_000_000))

    p1.start()
    p2.start()

    p1.join()
    p2.join()


if __name__ == '__main__':
    run_two_processes()


## 3. Process Pools (`Pool.map`)

`multiprocessing.Pool`, sabit sayıda işçi süreci (worker process) oluşturarak bir veri koleksiyonu üzerindeki bağımsız hesaplamaları mevcut CPU çekirdeklerine otomatik olarak dağıtan ve sonuçları sırayla toplayan havuz yapısıdır.

**Syntax**

```Python
with mp.Pool(processes=4) as pool:
    results = pool.map(func, iterable)
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **Varsayılan İşçi Sayısı:** `processes` parametresi belirtilmezse `os.process_cpu_count()` / `mp.cpu_count()` değerine eşit sayıda işçi süreci başlatılır.
* **Kapasite ve Kuyruklama:** İşlenecek öğe sayısı havuzdaki işçi sayısından fazlaysa, ilk grup tamamlanıp çekirdekler boşa çıktıkça kalan öğeler sıradaki boş işçilere atanır.
* **Bağlam Yöneticisi (`with`):** `with mp.Pool(...) as pool:` bloğu kullanıldığında işler tamamlandığında havuz otomatik kapatılır (`close()` ve `join()`).


In [ ]:
import multiprocessing as mp


def repeat_marker(count: int) -> str:
    cpu_burn(5_000_000)
    return 'X' * count


@measure_time
def run_pool_map() -> None:
    ctx = mp.get_context('fork') if 'fork' in mp.get_all_start_methods() else mp.get_context()
    inputs: tuple[int, ...] = tuple(range(1, 7))

    with ctx.Pool(processes=min(4, mp.cpu_count())) as pool:
        results: list[str] = pool.map(repeat_marker, inputs)

    print(results)


if __name__ == '__main__':
    run_pool_map()


## 4. Passing Multiple Arguments (`Pool.starmap`)

Standart `Pool.map()` hedef fonksiyona her seferinde yalnızca tek bir argüman geçirirken, `Pool.starmap()` demetlerden (tuple) oluşan bir iterable içindeki her öğeyi `*args` şeklinde açarak (unpack) çok parametreli veya değişken argümanlı (`*numbers`) fonksiyonlara aktarır.

**Syntax**

```python
with mp.Pool() as pool:
    results = pool.starmap(func, [(a1, b1), (a2, b2)])
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **Esnek Argüman Uzunlukları:** Hedef fonksiyon `*args` kabul ediyorsa, `starmap()` listesinde farklı uzunluklarda demetler (`(1, 2)`, `(3, 4, 10)`, `(5, 6, 7, 8)`) aynı havuz çağrısında işlenebilir.
* **`map` vs. `starmap`:** `map(fn, [(1, 2)])` çağrısı fonksiyona tek bir `tuple` nesnesi gönderirken, `starmap(fn, [(1, 2)])` çağrısı `fn(1, 2)` şeklinde iki ayrı argüman gönderir.


In [ ]:
import multiprocessing as mp


def sum_values(*numbers: float) -> float:
    return float(sum(numbers))


if __name__ == '__main__':
    ctx = mp.get_context('fork') if 'fork' in mp.get_all_start_methods() else mp.get_context()
    batches: list[tuple[float, ...]] = [
        (1.0, 2.0),
        (3.5, 4.5, 10.0),
        (5.0, 6.0),
        (7.0, 8.0, 111.0),
    ]

    with ctx.Pool(processes=min(4, mp.cpu_count())) as pool:
        totals: list[float] = pool.starmap(sum_values, batches)

    print(f'Totals: {totals}')


## 5. Running Multiple Functions in a Pool (`functools.partial`)

`Pool.map()` tek bir hedef fonksiyon kabul eder; farklı imzalara sahip birden fazla fonksiyonu aynı havuzda paralel çalıştırmak için her fonksiyon argümanlarıyla birlikte `functools.partial` kullanılarak sıfır parametreli çağrılabilir nesnelere (callable) dönüştürülür ve ortak bir tetikleyici fonksiyona verilir.

**Syntax**

```python
from functools import partial

job_a = partial(func_a, 'arg1')
job_b = partial(func_b, 'arg1', 'arg2')
results = pool.map(invoke_callable, [job_a, job_b])
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **Serileştirme (Pickling):** `lambda` ifadeleri `pickle` tarafından serileştirilemediği için `multiprocessing` havuzlarında kullanılamaz; `functools.partial` ise üst düzey fonksiyonlarla birlikte güvenle serileştirilebilir.
* **Heterojen İş Yükleri:** Farklı dönüş tiplerine veya parametre sayılarına sahip bağımsız görevleri tek bir `pool.map()` çağrısında toplamak için idealdir.


In [ ]:
import multiprocessing as mp
from collections.abc import Callable
from functools import partial
from typing import Any


def task_alpha(label: str) -> str:
    return f'Alpha:{label}'


def task_beta(value: int) -> int:
    return value * 10


def task_gamma(code: str, version: int) -> tuple[str, int]:
    return (code, version)


def run_callable(fn: Callable[[], Any]) -> Any:
    return fn()


if __name__ == '__main__':
    ctx = mp.get_context('fork') if 'fork' in mp.get_all_start_methods() else mp.get_context()
    jobs: list[Callable[[], Any]] = [
        partial(task_alpha, 'init'),
        partial(task_beta, 42),
        partial(task_gamma, 'v-core', 3),
    ]

    with ctx.Pool(processes=3) as pool:
        outputs: list[Any] = pool.map(run_callable, jobs)

    print(f'Mixed outputs: {outputs}')


## 6. Process Memory Isolation (Data Sharing Issue)

`threading` modülünün aksine, `multiprocessing` ile oluşturulan her alt süreç ana sürecin belleğinden tamamen izole edilmiş kendi adres alanına (isolated memory space) sahiptir.

**Dikkat Edilmesi Gerekenler**

* **Global Değişkenler Paylaşılmaz:** Bir alt süreç içinde `global` anahtar kelimesiyle yapılan değişiklikler yalnızca o alt sürecin kendi bellek kopyasını etkiler; ana süreçteki (main process) değişken değişmeden kalır.
* **Süreçler Arası İletişim (IPC):** Alt süreçlerden ana sürece veya süreçler arasında veri aktarmak için `Pipe`, `Queue`, `Value`/`Array` veya `multiprocessing.Manager` gibi IPC mekanizmaları kullanılmalıdır.


In [ ]:
import multiprocessing as mp

shared_items: list[int] = [0]


def mutate_in_child() -> None:
    global shared_items  # noqa: PLW0602
    shared_items.extend([1, 2, 3])
    print(f'Child process list: {shared_items}')


if __name__ == '__main__':
    ctx = mp.get_context('fork') if 'fork' in mp.get_all_start_methods() else mp.get_context()
    proc = ctx.Process(target=mutate_in_child)
    proc.start()
    proc.join()

    print(f'Main process list: {shared_items}')


## 7. Inter-Process Communication with Pipes (`mp.Pipe` Basics)

`multiprocessing.Pipe()`, iki süreç arasında serileştirilebilir (`pickleable`) Python nesnelerini göndermek ve almak için iki uçlu (`conn1, conn2`) bir iletişim kanalı oluşturur.

**Syntax**

```python
recv_conn, send_conn = mp.Pipe(duplex=False)
send_conn.send(payload)
has_data: bool = recv_conn.poll(timeout=0.5)
item = recv_conn.recv()
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **Tek Yönlü vs. Çift Yönlü (`duplex`):** Varsayılan `duplex=True` ayarında her iki uç da hem `send()` hem `recv()` yapabilir; `duplex=False` verildiğinde ise `conn1` yalnızca okuma (`recv`), `conn2` yalnızca yazma (`send`) ucu olarak sınırlandırılır.
* **Bloklayıcı `recv()` ve `poll()`:** `recv()` çağrısı kanalda veri yoksa veri gelene kadar süreci bloklar; veri olup olmadığını bloklamadan kontrol etmek için `poll(timeout)` kullanılır.
* **Eşzamanlı Erişim Uyarısı:** Aynı pipe ucuna iki farklı sürecin aynı anda yazması veya okuması veriyi bozabilir; ikiden fazla süreç varsa `Queue` tercih edilmelidir.


In [ ]:
import multiprocessing as mp

if __name__ == '__main__':
    receiver_conn, sender_conn = mp.Pipe(duplex=False)

    print(f'Pending before send: {receiver_conn.poll()}')
    sender_conn.send({'event': 'ready', 'code': 200})
    print(f'Pending after send: {receiver_conn.poll()}')

    if receiver_conn.poll(timeout=0.1):
        message: dict[str, str | int] = receiver_conn.recv()
        print(f'Received payload: {message}')

    print(f'Pending after recv: {receiver_conn.poll()}')
    sender_conn.close()
    receiver_conn.close()


## 8. Multi-Process Pipelines & Sentinel Values

İki ayrı süreç arasında `Pipe` ile sürekli veri akışı kurulduğunda, alıcı sürecin `while True:` döngüsünden ne zaman çıkacağını bilmesi için gönderici süreç tarafından özel bir bitiş sinyali (**sentinel value**, genellikle `None`) gönderilir.

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **Sentinel Deseni:** Alıcı taraf sonsuz bir beklemede (`recv()`) kilitlenip kalmasın diye üretici süreç tüm verileri gönderdikten sonra kanala `None` bırakır; alıcı `if data is None: break` kontrolüyle döngüyü güvenli şekilde sonlandırır.
* **Kaynak Temizliği:** Kullanılmayan uçların kapatılması (`conn.close()`) dosya tanıtıcılarının sızmasını önler.


In [ ]:
import multiprocessing as mp
import os
import random
import time
from multiprocessing.connection import Connection


def producer(conn: Connection) -> None:
    print(f'Producer PID: {os.getpid()}')
    for _ in range(4):
        value: int = random.randint(10, 99)
        conn.send(value)
        print(f'Sent -> {value}')
        time.sleep(0.1)

    conn.send(None)
    conn.close()


def consumer(conn: Connection) -> None:
    print(f'Consumer PID: {os.getpid()}')
    while True:
        item: int | None = conn.recv()
        if item is None:
            print('Sentinel received, exiting consumer.')
            break
        print(f'Received <- {item}')
    conn.close()


if __name__ == '__main__':
    ctx = mp.get_context('fork') if 'fork' in mp.get_all_start_methods() else mp.get_context()
    rx, tx = ctx.Pipe(duplex=False)

    p_send = ctx.Process(target=producer, args=(tx,))
    p_recv = ctx.Process(target=consumer, args=(rx,))

    p_send.start()
    p_recv.start()

    p_send.join()
    p_recv.join()


## 9. Inter-Process Queues (`mp.Queue` FIFO Basics)

`multiprocessing.Queue`, birden fazla üretici ve tüketici sürecin güvenli şekilde veri ekleyip (`put`) çıkarabileceği (`get`), süreç-güvenli (process-safe) ve ilk giren ilk çıkar (**FIFO** - First-In, First-Out) prensibiyle çalışan iletişim kuyruğudur[cite: 88].

**Syntax**

```python
q = mp.Queue()
q.put(item)
value = q.get()
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **FIFO ve Yürütme Sırası:** Kuyruğa eklenen öğeler eklenme sırasıyla çıkar; ancak öğeleri kuyruğa farklı süreçler ekliyorsa işletim sistemi zamanlaması nedeniyle tamamlanma ve kuyruğa eklenme sırası deterministik değildir[cite: 88].
* **Bloklama Riski:** Kuyrukta kalan öğe sayısından fazla `q.get()` çağrısı yapılırsa program yeni öğe gelene kadar süresiz bloklanır; bu nedenle üretilen öğe sayısı kadar okuma yapılmalı veya sentinel/timeout kullanılmalıdır[cite: 88].


In [ ]:
import multiprocessing as mp


def enqueue_worker(q: mp.Queue, value: int) -> None:
    q.put(value)
    print(f'Enqueued: {value}')


if __name__ == '__main__':
    ctx = mp.get_context('fork') if 'fork' in mp.get_all_start_methods() else mp.get_context()
    queue: mp.Queue = ctx.Queue()

    workers = [ctx.Process(target=enqueue_worker, args=(queue, i)) for i in range(5)]
    for w in workers:
        w.start()
    for w in workers:
        w.join()

    collected: list[int] = [queue.get() for _ in workers]
    print(f'Collected from queue: {collected}')


## 10. Queue Timeouts & Exception Handling (`queue.Empty`)

`Queue.get()` çağrısında kuyruk boşsa süresiz beklemeyi (deadlock) önlemek için `timeout` parametresi kullanılır; belirtilen süre içinde öğe gelmezse standart `queue.Empty` istisnası fırlatılır[cite: 89].

**Syntax**

```python
from queue import Empty

try:
    item = q.get(block=True, timeout=2.0)
except Empty:
    handle_timeout()
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **İstisna Türü:** `multiprocessing` kendi `Queue` sınıfını sunsa da fırlatılan zaman aşımı istisnası standart kütüphanedeki `queue.Empty` (ve dolu kuyruğa yazarken `queue.Full`) sınıfıdır.
* **Çoklu Tüketici:** Kuyruğa önceden sıralı eklenen veriler birden fazla işçi süreci tarafından çekildiğinde her işçi sıradaki ilk mevcut öğeyi alır ve işler[cite: 89].


In [ ]:
import multiprocessing as mp
from queue import Empty


def consume_with_timeout(q: mp.Queue) -> None:
    worker_name: str = mp.current_process().name
    try:
        item: int = q.get(timeout=0.5)
        print(f'{worker_name} received: {item}')
    except Empty:
        print(f'{worker_name} timed out waiting for data')


if __name__ == '__main__':
    ctx = mp.get_context('fork') if 'fork' in mp.get_all_start_methods() else mp.get_context()
    queue: mp.Queue = ctx.Queue()

    for num in (10, 20, 30):
        queue.put(num)

    consumers = [ctx.Process(target=consume_with_timeout, name=f'Consumer-{i}', args=(queue,)) for i in range(1, 5)]
    for c in consumers:
        c.start()
    for c in consumers:
        c.join()


## 11. Preserving Result Order with Indexed Tuples (`enumerate`)

Ham `Process` ve `Queue` kullanıldığında sonuçlar süreçlerin bitiş hızına göre karışık sırada kuyruğa girer; sonuçları orijinal girdi sırasıyla geri almak için girdiler `enumerate()` ile indekslenir ve kuyruğa `(index, result)` demeti olarak eklenir[cite: 90].

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **Sıralama Anahtarı:** Kuyruktan çekilen `(index, value)` demetleri `sorted()` fonksiyonuna verildiğinde Python demetleri ilk elemana (`index`) göre sıralar; ardından liste üreteciyle yalnızca `value` kısmı ayıklanır[cite: 90].
* **Alternatif:** Eğer yalnızca tek bir fonksiyon farklı girdilerle çalıştırılıyorsa `Pool.map()` bu sıralamayı dahili olarak otomatik yapar.


In [ ]:
import multiprocessing as mp
import random  # noqa: F811
import time  # noqa: F811


def compute_square(idx: int, number: int, q: mp.Queue) -> None:
    time.sleep(random.uniform(0.05, 0.25))
    q.put((idx, number ** 2))


if __name__ == '__main__':
    ctx = mp.get_context('fork') if 'fork' in mp.get_all_start_methods() else mp.get_context()
    queue: mp.Queue = ctx.Queue()
    numbers: list[int] = list(range(1, 9))

    workers = [
        ctx.Process(target=compute_square, args=(idx, num, queue))
        for idx, num in enumerate(numbers)
    ]
    for w in workers:
        w.start()
    for w in workers:
        w.join()

    raw_results: list[tuple[int, int]] = [queue.get() for _ in workers]
    ordered_squares: list[int] = [val for _, val in sorted(raw_results)]

    print(f'Unsorted pairs: {raw_results}')
    print(f'Ordered results: {ordered_squares}')


## 12. Process Synchronization (`mp.Lock` & `mp.Semaphore`)

`threading` modülündeki senkronizasyon ilkel yapıları `multiprocessing` altında da (`mp.Lock`, `mp.RLock`, `mp.Semaphore`, `mp.BoundedSemaphore`) süreçler arası ortak kaynak erişimini kontrol etmek için aynı `with` sözdizimiyle kullanılabilir[cite: 91].

**Syntax**

```python
lock = mp.Lock()
sem = mp.Semaphore(3)

with sem:
    access_shared_hardware_or_file()
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **Ortak Dış Kaynaklar:** Süreçler bellek paylaşmasa da aynı dosyaya yazarken, aynı seri porta/donanıma erişirken veya harici bir kaynağa eşzamanlı istek sınırlandırırken `Lock` (tek süreç) ve `Semaphore(n)` (en fazla `n` süreç) kullanılır[cite: 91].
* **Bağlam Yöneticisi:** `with` bloğu kullanmak kilitlerin istisna durumlarında dahi otomatik serbest bırakılmasını sağlar[cite: 91].


In [ ]:
import multiprocessing as mp
import time  # noqa: F811
from multiprocessing.synchronize import Semaphore as MpSemaphore


def limited_process(sem: MpSemaphore, worker_id: int) -> None:
    with sem:
        print(f'Process-{worker_id} entered critical section')
        time.sleep(0.2)
        print(f'Process-{worker_id} leaving critical section')


if __name__ == '__main__':
    ctx = mp.get_context('fork') if 'fork' in mp.get_all_start_methods() else mp.get_context()
    semaphore: MpSemaphore = ctx.Semaphore(3)

    procs = [ctx.Process(target=limited_process, args=(semaphore, i)) for i in range(1, 6)]
    for p in procs:
        p.start()
    for p in procs:
        p.join()
